Our goal is to build a recommendation system using text review.

In [ ]:
import pandas as pd

# Load the data
print("Loading datasets")
df_reviews = pd.read_csv('reviews83325.csv')
df_places = pd.read_csv('Tripadvisor.csv')

# Filter for English reviews only
print("Filtering for English reviews")
df_reviews_en = df_reviews[df_reviews['langue'] == 'en'].copy()

# Handle missing text and make everything lowercase
review_col_name = 'review' 
df_reviews_en[review_col_name] = df_reviews_en[review_col_name].fillna('').str.lower()

#  Group by idplace to combine all reviews for a place into one string
print("Aggregating reviews per place")
documents_df = df_reviews_en.groupby('idplace')[review_col_name].apply(lambda x: ' '.join(x)).reset_index()

# Rename the column for clarity
documents_df.rename(columns={review_col_name: 'aggregated_reviews'}, inplace=True)

# Merge with the places metadata so we know the names of the places
final_df = pd.merge(documents_df, df_places[['id', 'nom']], left_on='idplace', right_on='id', how='inner')

print(f"Data preparation complete! Total unique places: {len(final_df)}")

# Display the first few rows
final_df.head()

Loading datasets


C:\Users\natha\AppData\Local\Temp\ipykernel_28756\2518940421.py:5: DtypeWarning: Columns (15,16,17,18,19,20) have mixed types. Specify dtype option on import or set low_memory=False.
  df_reviews = pd.read_csv('reviews83325.csv')


Filtering for English reviews
Aggregating reviews per place
Data preparation complete! Total unique places: 1835


,idplace,aggregated_reviews,id,nom
0,188467,personally i think it is the most beautiful sq...,188467,Place des Vosges
1,188468,my old college friend and i booked this beauti...,188468,Rue des Francs Bourgeois
2,188470,"being winter and all, not a lot going on, howe...",188470,Village Saint-Paul
3,188471,to call au passe partout a shop is a serious u...,188471,Au Passe-partout
4,188472,very old historical place. i attended to exper...,188472,Cloître des Billettes


In [18]:
!pip install rank-bm25

In [19]:
from sklearn.model_selection import train_test_split
from rank_bm25 import BM25Okapi

print("Splitting data into 50% Train and 50% Test")
train_df, test_df = train_test_split(final_df, test_size=0.5, random_state=42)


def tokenize(text):
    
    return text.split()

print("Tokenizing the test corpus for BM25")
# We apply the tokenization to the test set 
tokenized_corpus = test_df['aggregated_reviews'].apply(tokenize).tolist()

print("Initializing the BM25 model")
# Build the BM25 index using the test corpus
bm25 = BM25Okapi(tokenized_corpus)

print("BM25 model is ready!")
print(f"Number of queries (train): {len(train_df)}")
print(f"Number of documents in corpus (test): {len(test_df)}")

Splitting data into 50% Train and 50% Test
Tokenizing the test corpus for BM25
Initializing the BM25 model
BM25 model is ready!
Number of queries (train): 917
Number of documents in corpus (test): 918


In [25]:
import numpy as np
from tqdm import tqdm 

print("Mapping metadata for Level 1 Evaluation")
place_to_typeR = dict(zip(df_places['id'], df_places['typeR']))

level_1_errors = []

print("Running Level 1 Evaluation for BM25")

# Loop through the train set
for index, query_row in tqdm(train_df.iterrows(), total=train_df.shape[0]):
    query_id = query_row['id']
    query_text = query_row['aggregated_reviews']
    
    # Get the true category for the query place
    query_type = place_to_typeR.get(query_id)
    if pd.isna(query_type):
        continue # Skip if missing
        
    # Tokenize query
    tokenized_query = tokenize(query_text)
    
    # Get BM25 scores 
    scores = bm25.get_scores(tokenized_query)
    
    # Sort test documents by score in descending order
    ranked_indices = np.argsort(scores)[::-1]
    
    # Find the rank of the first matching place
    for rank, test_idx in enumerate(ranked_indices):
        recommended_place_id = test_df.iloc[test_idx]['id']
        recommended_type = place_to_typeR.get(recommended_place_id)
        
        # If the categories match, record the error and stop looking
        if recommended_type == query_type:
            level_1_errors.append(rank) 
            break

# Calculate Average Error
if level_1_errors:
    avg_level_1_error = np.mean(level_1_errors)
    print(f"\nBM25 Average Level 1 Error: {avg_level_1_error:.2f}")
else:
    print("\nSomething went wrong. No matches found.")

Mapping metadata for Level 1 Evaluation
Running Level 1 Evaluation for BM25


100%|██████████| 917/917 [31:23<00:00,  2.05s/it]  


BM25 Average Level 1 Error: 0.94


In [21]:
import ast

print("Parsing Level 2 metadata from Tripadvisor.csv")

# Helper function to safely parse string lists from the CSV
def parse_list_column(val):
    if pd.isna(val):
        return []
    if isinstance(val, str):
        try:
            parsed = ast.literal_eval(val)
            if isinstance(parsed, list):
                return parsed
            else:
                return [val]
        except (ValueError, SyntaxError):
            return [x.strip() for x in val.split(',')]
    return [val]

# Dictionary to store all our Level 2 data
level_2_metadata = {}

for index, row in df_places.iterrows():
    place_id = row['id']
    typeR = row['typeR']
    
    # We will gather all relevant Level 2 attributes into a single flat list for easy intersection checking
    attributes = []
    
    if typeR == 'R': # Restaurant 
        attributes.extend(parse_list_column(row.get('restaurantType')))
        attributes.extend(parse_list_column(row.get('cuisine')))
        
    elif typeR in ['A', 'AP']: 
        attributes.extend(parse_list_column(row.get('activiteSubCategorie')))
        attributes.extend(parse_list_column(row.get('activiteSubType')))
        
    elif typeR == 'H': # Hotel 
        price = row.get('priceRange')
        if pd.notna(price):
            attributes.append(price)
            
    # Save it to our lookup dictionary
    level_2_metadata[place_id] = {
        'typeR': typeR,
        'attributes': set(attributes)
    }

print("Level 2 metadata dictionary is ready!")

Parsing Level 2 metadata from Tripadvisor.csv
Level 2 metadata dictionary is ready!


In [22]:
print("Running Level 2 Evaluation for BM25")

level_2_errors = []
queries_without_metadata = 0

# Loop through the train set 
for index, query_row in tqdm(train_df.iterrows(), total=train_df.shape[0]):
    query_id = query_row['id']
    query_text = query_row['aggregated_reviews']
    
    # Grab the Level 2 metadata for our query place
    query_meta = level_2_metadata.get(query_id)
    
    # If the place isn't in our dictionary or has absolutely no attributes to match against, skip it
    if not query_meta or len(query_meta['attributes']) == 0:
        queries_without_metadata += 1
        continue
        
    query_typeR = query_meta['typeR']
    query_attributes = query_meta['attributes'] 
    
    # Tokenize query
    tokenized_query = tokenize(query_text)
    
    # Get BM25 scores for all test documents
    scores = bm25.get_scores(tokenized_query)
    
    # Sort test documents by score descending
    ranked_indices = np.argsort(scores)[::-1]
    
    # Find the rank of the first matching place
    match_found = False
    
    for rank, test_idx in enumerate(ranked_indices):
        recommended_place_id = test_df.iloc[test_idx]['id']
        recommended_meta = level_2_metadata.get(recommended_place_id)
        
        if not recommended_meta:
            continue
            
        recommended_typeR = recommended_meta['typeR']
        recommended_attributes = recommended_meta['attributes']
        
        # LEVEL 2 CHECK: 
        # Must be the same type of place 
        # The intersection of their attribute sets must be greater than 0
        if (recommended_typeR == query_typeR) and (len(query_attributes.intersection(recommended_attributes)) > 0):
            level_2_errors.append(rank)
            match_found = True
            break
            
    # If no match is found in the entire set, the error is technically "undefined"

# 5. Calculate Average Error
if level_2_errors:
    avg_level_2_error = np.mean(level_2_errors)
    print(f"\nBM25 Average Level 2 Error: {avg_level_2_error:.2f}")
    print(f"Total queries evaluated: {len(level_2_errors)}")
    print(f"Queries skipped (no metadata to match against): {queries_without_metadata}")
else:
    print("\nSomething went wrong. No matches found.")

Running Level 2 Evaluation for BM25


100%|██████████| 917/917 [29:16<00:00,  1.92s/it]  


BM25 Average Level 2 Error: 15.40
Total queries evaluated: 380
Queries skipped (no metadata to match against): 503


BM25 is our baseline : Level 1 = 0.94 | Level 2 = 15.40. It works by searching for exact word matching. Many queries were skipped in the level 2 error because of that.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from tqdm import tqdm

print("Initializing TF-IDF Vectorizer")
# We use TF-IDF to extract the most important words, ignoring common English stop words.
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)

print("Vectorizing the Test Corpus")
test_texts = test_df['aggregated_reviews'].tolist()
# This creates our mathematical matrix for the test set
test_tfidf_matrix = vectorizer.fit_transform(test_texts)

print("Starting Evaluation for TF-IDF Model")

level_1_errors_tfidf = []
level_2_errors_tfidf = []

# Loop through our train_df 
for index, query_row in tqdm(train_df.iterrows(), total=train_df.shape[0]):
    query_id = query_row['id']
    query_text = query_row['aggregated_reviews']
    
    # Get metadata
    query_typeR = place_to_typeR.get(query_id)
    query_meta_l2 = level_2_metadata.get(query_id)
    
    if pd.isna(query_typeR):
        continue
        
    # Vectorize the query text using the same vectorizer
    query_vector = vectorizer.transform([query_text])
    
    # Calculate Cosine Similarity between the query and all test documents
    similarity_scores = cosine_similarity(query_vector, test_tfidf_matrix).flatten()
    
    # Sort test documents by similarity score
    ranked_indices = np.argsort(similarity_scores)[::-1]
    
    found_l1 = False
    found_l2 = False
    
    for rank, test_idx in enumerate(ranked_indices):
        recommended_place_id = test_df.iloc[test_idx]['id']
        
        # Level 1 Check
        if not found_l1:
            if place_to_typeR.get(recommended_place_id) == query_typeR:
                level_1_errors_tfidf.append(rank)
                found_l1 = True
                
        # Level 2 Check
        if not found_l2 and query_meta_l2 and len(query_meta_l2['attributes']) > 0:
            rec_meta_l2 = level_2_metadata.get(recommended_place_id)
            if rec_meta_l2:
                if (rec_meta_l2['typeR'] == query_typeR) and \
                   (len(query_meta_l2['attributes'].intersection(rec_meta_l2['attributes'])) > 0):
                    level_2_errors_tfidf.append(rank)
                    found_l2 = True
                    
        # Stop looking if we found both matches for this query
        if found_l1 and (found_l2 or not query_meta_l2 or len(query_meta_l2['attributes']) == 0):
            break

# Print the results
print("\nTF-IDF MODEL RESULTS")
if level_1_errors_tfidf:
    print(f"Average Level 1 Error: {np.mean(level_1_errors_tfidf):.2f}")
if level_2_errors_tfidf:
    print(f"Average Level 2 Error: {np.mean(level_2_errors_tfidf):.2f}")

Initializing TF-IDF Vectorizer
Vectorizing the Test Corpus
Starting Evaluation for TF-IDF Model


100%|██████████| 917/917 [00:15<00:00, 60.91it/s]


TF-IDF MODEL RESULTS
Average Level 1 Error: 0.92
Average Level 2 Error: 6.33


TF-IDF: Level 1 = 0.92 | Level 2 = 6.33
This model is a big upgrade from BM25. We obtain this by limiting the vocabulary to the 5000 most important words and using the TF-IDF method. 

In [24]:
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from tqdm import tqdm

print("Applying LSA (Latent Semantic Analysis)")
# We compress our 5000 TF-IDF features down to 100 hidden "concepts"
lsa_model = TruncatedSVD(n_components=100, random_state=42)

# Transform our test set from TF-IDF words into LSA concept vectors
test_lsa_matrix = lsa_model.fit_transform(test_tfidf_matrix)

print("Starting Evaluation for LSA Model")

level_1_errors_lsa = []
level_2_errors_lsa = []

# Loop through our train_df
for index, query_row in tqdm(train_df.iterrows(), total=train_df.shape[0]):
    query_id = query_row['id']
    query_text = query_row['aggregated_reviews']
    
    query_typeR = place_to_typeR.get(query_id)
    query_meta_l2 = level_2_metadata.get(query_id)
    
    if pd.isna(query_typeR):
        continue
        
    # get the TF-IDF vector for the query
    query_tfidf = vectorizer.transform([query_text])
    
    # Compress the query into the same 100 LSA concepts
    query_lsa = lsa_model.transform(query_tfidf)
    
    # Calculate Cosine Similarity on the LSA vectors
    similarity_scores = cosine_similarity(query_lsa, test_lsa_matrix).flatten()
    
    # Sort documents by score
    ranked_indices = np.argsort(similarity_scores)[::-1]
    
    found_l1 = False
    found_l2 = False
    
    for rank, test_idx in enumerate(ranked_indices):
        recommended_place_id = test_df.iloc[test_idx]['id']
        
        # Level 1 Check
        if not found_l1:
            if place_to_typeR.get(recommended_place_id) == query_typeR:
                level_1_errors_lsa.append(rank)
                found_l1 = True
                
        # Level 2 Check
        if not found_l2 and query_meta_l2 and len(query_meta_l2['attributes']) > 0:
            rec_meta_l2 = level_2_metadata.get(recommended_place_id)
            if rec_meta_l2:
                if (rec_meta_l2['typeR'] == query_typeR) and \
                   (len(query_meta_l2['attributes'].intersection(rec_meta_l2['attributes'])) > 0):
                    level_2_errors_lsa.append(rank)
                    found_l2 = True
                    
        if found_l1 and (found_l2 or not query_meta_l2 or len(query_meta_l2['attributes']) == 0):
            break

# Print the results
print("\nLSA MODEL RESULTS")
if level_1_errors_lsa:
    print(f"Average Level 1 Error: {np.mean(level_1_errors_lsa):.2f}")
if level_2_errors_lsa:
    print(f"Average Level 2 Error: {np.mean(level_2_errors_lsa):.2f}")

Applying LSA (Latent Semantic Analysis)
Starting Evaluation for LSA Model


100%|██████████| 917/917 [00:10<00:00, 90.83it/s] 


LSA MODEL RESULTS
Average Level 1 Error: 0.70
Average Level 2 Error: 5.32


LSA : Level 1 = 0.70 | Level 2 = 5.32
This model goes beyond the TF-IDF going from 5000 words to 100 concepts. This model understands synonyms and topics. This is why this one performs the best.